# 01 — Download, preprocess and EDA

SE4050 Deep Learning. **Run this once, by the group leader.**

Produces `train.csv`, `val.csv`, `test.csv` and `split_meta.json`, which every
model then loads. Nobody else re-splits or re-cleans.

Runtime → Change runtime type → **T4 GPU** (not needed here, but keeps the
session alive longer).

**Order:** setup → download → subset → split → EDA → save to Drive.

## 1. Setup — clone the repo and install

In [ ]:
!git clone -q https://github.com/ravindu912/Deep-Learning-Project.git
%cd Deep-Learning-Project
!pip install -q pandas scikit-learn matplotlib seaborn

import sys, os
sys.path.insert(0, os.getcwd())
print('working dir:', os.getcwd())

## 2. Download the CFPB database

About 2–3 GB zipped and it takes a few minutes. **Write down the date** — the
database updates daily, and the report must state which snapshot you used.

In [ ]:
from datetime import date
DOWNLOAD_DATE = date.today().isoformat()
print('Snapshot date (put this in the report):', DOWNLOAD_DATE)

!mkdir -p data
!wget -q --show-progress -O data/complaints.csv.zip https://files.consumerfinance.gov/ccdb/complaints.csv.zip
!ls -lh data/

In [ ]:
import zipfile, pandas as pd

with zipfile.ZipFile('data/complaints.csv.zip') as z:
    name = z.namelist()[0]
    print('extracting', name)
    z.extract(name, 'data/')

RAW = f'data/{name}'

# Read only the two columns we need — the full file has 18 and will
# exhaust Colab's RAM if loaded whole.
COLS = ['Consumer complaint narrative', 'Product']
raw = pd.read_csv(RAW, usecols=COLS, low_memory=False)
print(f'{len(raw):,} total complaints')
raw.head(3)

## 3. Build the working subset

Only complaints where the consumer agreed to publish the narrative have text.
We cap each class so DistilBERT trains in about an hour on a free GPU.

In [ ]:
PER_CLASS = 15000   # lower this to 8000 if training is too slow
SEED = 42

df = raw.dropna(subset=['Consumer complaint narrative'])
print(f'with a narrative: {len(df):,} of {len(raw):,} ({len(df)/len(raw):.1%})')

sub = (df.groupby('Product', group_keys=False)
         .apply(lambda g: g.sample(min(len(g), PER_CLASS), random_state=SEED))
         .reset_index(drop=True))

sub.to_csv('data/complaints_subset.csv', index=False)
print(f'subset: {len(sub):,} rows -> data/complaints_subset.csv')
sub['Product'].value_counts()

## 4. The shared split

`src/data.py` does the cleaning, merges renamed product labels, removes
duplicates **before** splitting (the leakage guard), and writes the
stratified 70/15/15 split with seed 42.

In [ ]:
!python src/data.py --input data/complaints_subset.csv --out data/

In [ ]:
from src.data import load_splits

train, val, test, meta = load_splits('data')
print(f"train {len(train):,} | val {len(val):,} | test {len(test):,}")
print(f"classes ({meta['n_classes']}):")
for c in meta['classes']:
    print('  -', c)

# Leakage check — this must print 0, 0.
print('\ntrain∩test overlap:', len(set(train.text) & set(test.text)))
print('train∩val overlap :', len(set(train.text) & set(val.text)))

## 5. EDA — this is Section 3 of the report

Save every figure. The rubric wants visualisations *and* their interpretation.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

os.makedirs('results/figures', exist_ok=True)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3})

# --- class distribution ---
counts = train['label'].value_counts()
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh([c[:40] for c in counts.index][::-1], counts.values[::-1], color='#4269d0')
ax.set_xlabel('complaints in the training set')
ax.set_title('Class distribution (training set)')
fig.tight_layout(); fig.savefig('results/figures/class_distribution.png', bbox_inches='tight')

imbalance = counts.max() / counts.min()
print(f'imbalance ratio (largest:smallest) = {imbalance:.1f} : 1')
print('-> this is why macro F1 is the headline metric, not accuracy')

In [ ]:
# --- narrative length ---
lengths = train['text'].str.split().str.len()

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(lengths, bins=60, range=(0, 800), color='#4269d0')
for pct, colour in [(50, '#efb118'), (90, '#ff725c'), (95, '#6cc5b0')]:
    v = np.percentile(lengths, pct)
    ax.axvline(v, color=colour, ls='--', label=f'p{pct} = {v:.0f}')
ax.set_xlabel('words per complaint'); ax.set_ylabel('complaints')
ax.set_title('Narrative length (training set)'); ax.legend()
fig.tight_layout(); fig.savefig('results/figures/length_distribution.png', bbox_inches='tight')

print(f'median {lengths.median():.0f} | p90 {np.percentile(lengths,90):.0f} | max {lengths.max()}')
print('-> use the p90 value as MAX_LEN. Longer costs compute for few extra words.')

In [ ]:
# --- length by class: do some categories produce longer complaints? ---
order = counts.index.tolist()
data = [train.loc[train.label == c, 'text'].str.split().str.len() for c in order]

fig, ax = plt.subplots(figsize=(8, 5))
ax.boxplot(data, vert=False, labels=[c[:34] for c in order], showfliers=False)
ax.set_xlabel('words per complaint')
ax.set_title('Complaint length by product')
fig.tight_layout(); fig.savefig('results/figures/length_by_class.png', bbox_inches='tight')

In [ ]:
# --- top distinguishing words per class (for the report's discussion) ---
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=20000, stop_words='english', min_df=5)
X = tfidf.fit_transform(train['text'])
vocab = np.array(tfidf.get_feature_names_out())

for c in order:
    mask = (train['label'] == c).values
    score = np.asarray(X[mask].mean(0)).ravel() - np.asarray(X[~mask].mean(0)).ravel()
    print(f"{c[:44]:46s} {', '.join(vocab[score.argsort()[-8:][::-1]])}")

In [ ]:
# --- sample complaints: read a few, they explain the errors you'll see later ---
for c in order[:4]:
    row = train[train.label == c].iloc[0]
    print(f"[{c}]\n{row.text[:320]}...\n")

## 6. Save the splits to Drive and share them

Everyone must train on **these exact files**. They are gitignored, so they do
not travel through GitHub.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DEST = '/content/drive/MyDrive/SE4050_shared'
!mkdir -p "{DEST}"
!cp data/train.csv data/val.csv data/test.csv data/split_meta.json "{DEST}/"
!cp -r results/figures "{DEST}/"
!ls -lh "{DEST}"

print('\nShare this folder with the group. Message them: "Split is up, start your models."')

## Notes for the report

Fill these in from the output above:

| Item | Value |
| --- | --- |
| Snapshot date | |
| Total complaints in database | |
| With a published narrative | |
| After cleaning and dedupe | |
| Classes kept | |
| Imbalance ratio | |
| MAX_LEN chosen (p90) | |
| Train / val / test sizes | |

Data-quality issues to mention: personal details masked as `XXXX`, duplicate
submissions, renamed product categories over time, heavy class imbalance, and
narratives ranging from one line to several pages.